In [1]:
pip install httpx

Note: you may need to restart the kernel to use updated packages.


In [10]:
import httpx

URL = "https://httpbin.org/get"
# Fixed the headers dictionary - separated key and value properly with colon and added missing quote
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"}

def fetch_page(url: str) -> None:
    response = httpx.get(url, headers=headers)
    print(f"Status code:{response.status_code}")
    print(f"Content-Type: {response.headers['content-type']}")
    print(f"Body Preview: {response.text[:200]}") 
    print(f"User-Agent: {response.json()['headers']['User-Agent']}")

if __name__ == "__main__":
    fetch_page(URL)

Status code:200
Content-Type: application/json
Body Preview: {
  "args": {}, 
  "headers": {
    "Accept": "*/*", 
    "Accept-Encoding": "gzip, deflate, br, zstd", 
    "Host": "httpbin.org", 
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleW
User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36


In [11]:
pip install selectolax

   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.9 MB 1.9 MB/s eta 0:00:01
   ---------------------- ----------------- 1.0/1.9 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------  1.8/1.9 MB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 1.9/1.9 MB 2.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [15]:
#HTML Parsing with Selectolax
import httpx
from selectolax.parser import HTMLParser

URL = "https://books.toscrape.com/"  # A real sandbox site built for scraping practice

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
}

def scrape_books(url: str) -> None:
    response = httpx.get(url, headers=headers)
    tree = HTMLParser(response.text)      # Parse the HTML into a navigable tree

    # Each book lives inside an <article class="product_pod">
    books = tree.css("article.product_pod")
    print(f"Books found on page: {len(books)}")

    # Look at the first book only
    for index, book in enumerate(books, start=1):
        title = book.css_first("h3 a").attributes["title"]
        price = book.css_first(".price_color").text()
        print(f"{index}. {title} — {price}")

if __name__ == "__main__":
    scrape_books(URL)

Books found on page: 20
1. A Light in the Attic — £51.77
2. Tipping the Velvet — £53.74
3. Soumission — £50.10
4. Sharp Objects — £47.82
5. Sapiens: A Brief History of Humankind — £54.23
6. The Requiem Red — £22.65
7. The Dirty Little Secrets of Getting Your Dream Job — £33.34
8. The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull — £17.93
9. The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics — £22.60
10. The Black Maria — £52.15
11. Starving Hearts (Triangular Trade Trilogy, #1) — £13.99
12. Shakespeare's Sonnets — £20.66
13. Set Me Free — £17.46
14. Scott Pilgrim's Precious Little Life (Scott Pilgrim #1) — £52.29
15. Rip it Up and Start Again — £35.02
16. Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991 — £57.25
17. Olio — £23.88
18. Mesaerion: The Best Science Fiction Stories 1800-1849 — £37.59
19. Libertarianism for Beginners — £51.33
20. It's Only the Himalayas — £45.1

In [18]:
# Multi-Page Scraping (Pagination)
import httpx
from selectolax.parser import HTMLParser

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
}

def scrape_page(url: str) -> tuple[list[dict], bool]:
    """Scrape one page. Returns (books_data, has_next_page)."""
    response = httpx.get(url, headers=headers)
    tree = HTMLParser(response.text)

    books = []
    for node in tree.css("article.product_pod"):
        books.append({
            "title": node.css_first("h3 a").attributes["title"],
            "price": node.css_first(".price_color").text(),
        })

    # Check if a "next" button exists — if not, we're on the last page
    has_next = tree.css_first("li.next") is not None
    return books, has_next


def scrape_all_books() -> None:
    all_books = []
    page = 1
    total = 0

    while True:
        url = BASE_URL.format(page)
        books, has_next = scrape_page(url)
        all_books.extend(books)
        
        total += len(books)
        print(f"Page {page:>2} — scraped {len(books)} books (running total: {total})")

        if not has_next:
            print(f"\nDone! Scraped {total} books across {page} pages.")
            break

        page += 1
        
    sorted_books = sorted(
        all_books,
        key=lambda book: float(book["price"].replace("£", "")),
        reverse=True
    )
    top_3 = sorted_books[:3]
    
    print("\nTop 3 most expensive books:")
        for index, book in enumerate(top_3, start=1):
            print(f"{index}. {book['title']} — {book['price']}")

if __name__ == "__main__":
    scrape_all_books()

IndentationError: unexpected indent (1419384338.py, line 55)

In [20]:
#Top 3 most expensive books:
# Multi-Page Scraping (Pagination)
import httpx
from selectolax.parser import HTMLParser

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
}

def scrape_page(url: str) -> tuple[list[dict], bool]:
    """Scrape one page. Returns (books_data, has_next_page)."""
    response = httpx.get(url, headers=headers)
    tree = HTMLParser(response.text)

    books = []
    for node in tree.css("article.product_pod"):
        books.append({
            "title": node.css_first("h3 a").attributes["title"],
            "price": node.css_first(".price_color").text(),
        })

    has_next = tree.css_first("li.next") is not None
    return books, has_next


def scrape_all_books() -> None:
    all_books = []
    page = 1
    total = 0

    while True:
        url = BASE_URL.format(page)
        books, has_next = scrape_page(url)
        all_books.extend(books)
        total += len(books)
        print(f"Page {page:>2} — scraped {len(books)} books (running total: {total})")

        if not has_next:
            print(f"\nDone! Scraped {total} books across {page} pages.")
            break

        page += 1

    # All 1000 books collected — now sort by price
    sorted_books = sorted(
        all_books,
        key=lambda book: float(book["price"].replace("£", "")),
        reverse=True
    )

    top_3 = sorted_books[:3]

    print("\nTop 3 most expensive books:")
    for index, book in enumerate(top_3, start=1):
        print(f"{index}. {book['title']} — {book['price']}")


if __name__ == "__main__":
    scrape_all_books()

Page  1 — scraped 20 books (running total: 20)
Page  2 — scraped 20 books (running total: 40)
Page  3 — scraped 20 books (running total: 60)
Page  4 — scraped 20 books (running total: 80)
Page  5 — scraped 20 books (running total: 100)
Page  6 — scraped 20 books (running total: 120)
Page  7 — scraped 20 books (running total: 140)
Page  8 — scraped 20 books (running total: 160)
Page  9 — scraped 20 books (running total: 180)
Page 10 — scraped 20 books (running total: 200)
Page 11 — scraped 20 books (running total: 220)
Page 12 — scraped 20 books (running total: 240)
Page 13 — scraped 20 books (running total: 260)
Page 14 — scraped 20 books (running total: 280)
Page 15 — scraped 20 books (running total: 300)
Page 16 — scraped 20 books (running total: 320)
Page 17 — scraped 20 books (running total: 340)
Page 18 — scraped 20 books (running total: 360)
Page 19 — scraped 20 books (running total: 380)
Page 20 — scraped 20 books (running total: 400)
Page 21 — scraped 20 books (running total: 4